# BusNet — quick test: phase-coded multiplexing on a shared bus

Six object modules answer **two pairwise questions at once** ("is A left
of B", "is C above D", "do E and F share a shape"; disjoint pairs). Each
question is answered by a **head module** that holds only the question
and, like everyone else, hears nothing but the bus, so the answer can
only be assembled from what the two named objects broadcast at the
head's phase. The medium is a single **bus**: every module writes $m_j e^{i\theta_j}$, every receiver gets the
sum and demodulates with its own phase,

$$r_i = \mathrm{Re}\big(e^{-i\theta_i}\textstyle\sum_j m_j e^{i\theta_j}\big) = \sum_j m_j\cos(\theta_j-\theta_i).$$

There is no per-sender access, so attention cannot exist on the bus; the
only way to separate the two conversations is to put them $90°$ apart
(an antiphase sender arrives negated, an orthogonal one cancels), and two
orthogonal groups is the capacity of $S^1$: a third conversation would
leak by $\cos 60° = 0.5$.
The bus is one-dimensional by default so the two conversations cannot
be split by channel dimension either.

Predictions: `bus/open` (all phases zero, a plain sum) pays interference;
`bus/phase` recovers it with two orthogonal groups (`same_q_align` high,
`cross_q_align` near zero); `channels/attn` (per-sender access restored)
is the upper bound; widening the bus (`msg_dim` 2, 4) should let the open
bus catch up. `asker` is the private-line readout for comparison
(the answering module's state read directly).


In [ ]:
from __future__ import annotations

from pathlib import Path
import sys, os
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'main.py').exists())
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)

import torch
import torch.nn as nn

import time, math
import numpy as np

from src.tasks.sort_of_clevr.data.pairwise import make_pairwise_dataset, make_pairwise_from_npz, to_tensors, PAIR_SUBTYPES
from src.tasks.sort_of_clevr.data import constants as C
from src.models.busnet import BusNet, BusNetConfig

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
OUT_DIR = ROOT / 'notebooks' / 'outputs' / 'busnet_dev'
OUT_DIR.mkdir(parents=True, exist_ok=True)
COLOURS = list(C.COLOURS.values())


In [ ]:
N_TRAIN, N_TEST, N_Q = 20_000, 2_000, 2

# Scenes: reuse the stored splits (same scenes as every other model; only the
# questions are new) when the dev dataset exists, else generate and cache.
DATA_DIR = Path(r'/home/nik/workspace/ImperialWork/msc_project/SyncNetProject/data') / 'sort-of-clevr-notebook-dev'

def load_or_make(split: str, n: int, seed: int):
    npz = DATA_DIR / f'{split}.npz'
    if npz.exists():
        return make_pairwise_from_npz(npz, n_questions=N_Q, seed=seed, max_scenes=n)
    path = OUT_DIR / f'pairwise_q{N_Q}_{split}_{n}.npz'
    if path.exists():
        return dict(np.load(path))
    data = make_pairwise_dataset(n, n_questions=N_Q, seed=seed)
    np.savez_compressed(path, **data)
    return data

train_data = load_or_make('train', N_TRAIN, seed=0)
test_data = load_or_make('test', N_TEST, seed=1)
x_tr, q_tr, y_tr = to_tensors(train_data, DEVICE)
x_te, q_te, y_te = to_tensors(test_data, DEVICE)
sub_te = q_te[..., C.SUB_Q_TYPE_IDX:C.SUB_Q_TYPE_IDX + 3].argmax(-1)
print('train', tuple(x_tr.shape), tuple(q_tr.shape), '| test', tuple(x_te.shape),
      '| yes-rate', round(float((y_tr == 0).float().mean()), 3))


In [ ]:
def evaluate(model, x, q, y, **overrides):
    model.eval()
    hits, metrics = [], []
    with torch.no_grad(), torch.autocast(DEVICE, dtype=torch.bfloat16, enabled=DEVICE == 'cuda'):
        for i in range(0, len(x), 1024):
            out = model(x[i:i + 1024], q[i:i + 1024], **overrides)
            hits.append(out['logits'].argmax(-1) == y[i:i + 1024])
            metrics.append(out['metrics'])
    hit = torch.cat(hits)
    agg = {k: float(np.nanmean([m[k] for m in metrics])) for k in metrics[0]}
    per_sub = {PAIR_SUBTYPES[s]: hit[sub_te == s].float().mean().item() for s in range(3)}
    return hit.float().mean().item(), per_sub, agg


def train_busnet(name: str, steps: int = 3_000, bs: int = 256, lr: float = 1e-3, warmup: int = 200,
                 seed: int = 0, **kw):
    torch.manual_seed(seed)
    model = BusNet(BusNetConfig(name='busnet', n_questions=N_Q, **kw), 75, 2, COLOURS).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: min(1.0, (s + 1) / warmup) * 0.5 * (1 + math.cos(math.pi * min(s, steps) / steps)))
    n, t0, hist = len(x_tr), time.time(), []
    for step in range(steps):
        model.train()
        idx = torch.randint(0, n, (bs,), device=DEVICE)
        with torch.autocast(DEVICE, dtype=torch.bfloat16, enabled=DEVICE == 'cuda'):
            out = model(x_tr[idx], q_tr[idx])
            loss = nn.functional.cross_entropy(out['logits'].reshape(-1, 2).float(), y_tr[idx].reshape(-1))
        opt.zero_grad(set_to_none=True); loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step(); sched.step()
        if (step + 1) % 500 == 0 or step == steps - 1:
            acc, _, m = evaluate(model, x_te, q_te, y_te)
            hist.append((step + 1, loss.item(), acc))
            print(f'  {name:<22} step {step + 1:>5} loss {loss.item():.3f} test acc {acc:.3f} '
                  f'R {m["phase_R"]:.2f} same {m["same_q_align"]:+.2f} cross {m["cross_q_align"]:+.2f}')
    acc, per_sub, m = evaluate(model, x_te, q_te, y_te)
    drops = {po: acc - evaluate(model, x_te, q_te, y_te, phase_override=po)[0] for po in ['zero', 'shuffle', 'freeze']}
    print(f'  done in {time.time() - t0:.0f}s')
    return dict(acc=acc, per_sub=per_sub, metrics=m, drops=drops, hist=hist, model=model)


In [ ]:
ARMS = {
    'bus/zero':          dict(bus_phase='zero'),                       # no communication: the floor
    'channels/attn':     dict(medium='channels', gate_mode='attn'),    # per-sender access: the ceiling
    'bus/open msg1':     dict(bus_phase='open', msg_dim=1),            # plain sum: interference
    'bus/phase msg1':    dict(bus_phase='phase', msg_dim=1),           # phase demodulation, coupling only
    'bus/phase+stim msg1': dict(bus_phase='phase', msg_dim=1, drive='stimulus'),
    'bus/open msg2':     dict(bus_phase='open', msg_dim=2),            # can the open bus split by dimension?
    'bus/open msg4':     dict(bus_phase='open', msg_dim=4),
    'bus/phase msg4':    dict(bus_phase='phase', msg_dim=4),
    'bus/phase asker':   dict(bus_phase='phase', msg_dim=1, readout='asker'),   # private-line readout comparator
}

runs = {}
for name, kw in ARMS.items():
    print(f'\n===== {name} =====')
    runs[name] = train_busnet(name, **kw)


In [ ]:
print(f'{"arm":<22}{"acc":>7}' + ''.join(f'{s:>12}' for s in PAIR_SUBTYPES)
      + f'{"R":>7}{"same":>7}{"cross":>7}{"h_own":>7}{"h_oth":>7}{"n_cl":>6}{"d_zero":>8}{"d_shuf":>8}{"d_frz":>8}')
for name, r in runs.items():
    m, d = r['metrics'], r['drops']
    print(f'{name:<22}{r["acc"]:>7.3f}' + ''.join(f'{r["per_sub"][s]:>12.3f}' for s in PAIR_SUBTYPES)
          + f'{m["phase_R"]:>7.2f}{m["same_q_align"]:>+7.2f}{m["cross_q_align"]:>+7.2f}'
          + f'{m.get("head_own_align", float("nan")):>+7.2f}{m.get("head_other_align", float("nan")):>+7.2f}{m["n_clusters_eff"]:>6.2f}'
          + f'{d["zero"]:>+8.3f}{d["shuffle"]:>+8.3f}{d["freeze"]:>+8.3f}')
print('\nchance is .5; bus/zero is the no-communication floor; h_own / h_oth = cos alignment of each head')
print('with the pair it asks about / with the other pair (want +1 / ~0);')
print('channels/attn is the ceiling. d_zero = accuracy lost when every phase is set to 0 (open bus) at test time.')


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))
names = list(runs)
axes[0].bar(range(len(names)), [runs[n]['acc'] for n in names], color='tab:blue')
axes[0].axhline(0.5, color='grey', ls='--', lw=1); axes[0].set_ylim(0.4, 1.0)
axes[0].set_xticks(range(len(names))); axes[0].set_xticklabels(names, rotation=60, ha='right', fontsize=8)
axes[0].set_title('test accuracy', fontsize=10)
w = 0.4
axes[1].bar(np.arange(len(names)) - w / 2, [runs[n]['metrics']['same_q_align'] for n in names], w, label='same question')
axes[1].bar(np.arange(len(names)) + w / 2, [runs[n]['metrics']['cross_q_align'] for n in names], w, label='different question')
axes[1].axhline(0, color='grey', lw=1); axes[1].set_ylim(-1, 1); axes[1].legend(fontsize=8)
axes[1].set_xticks(range(len(names))); axes[1].set_xticklabels(names, rotation=60, ha='right', fontsize=8)
axes[1].set_title('mean cos(theta_i - theta_j) between named modules', fontsize=10)
# phase trajectories of the phase arm, three test scenes: named modules solid, idle dotted
r = runs['bus/phase msg1']; model = r['model'].eval()
with torch.no_grad():
    tr = model(x_te[:3], q_te[:3], return_trace=True)['traces']
ph = torch.stack(tr['phase']).float().cpu()                     # (T, B, N, 2): objects then heads
ang = torch.atan2(ph[..., 1], ph[..., 0]).numpy()
named = ((q_te[:3, :, :6] + q_te[:3, :, 6:12]) > 0).float().cpu().numpy()   # (B, Nq, M)
for b in range(3):
    for k in range(ph.shape[2]):
        if k < 6:
            grp = int(named[b, :, k].argmax()) if named[b, :, k].any() else -1
            axes[2].plot(np.unwrap(ang[:, b, k]), color=['tab:red', 'tab:blue', 'grey'][grp], lw=1,
                         ls=['-', '--', ':'][b], alpha=0.8)
        else:                                                    # head of question k-6, thick line
            axes[2].plot(np.unwrap(ang[:, b, k]), color=['tab:red', 'tab:blue'][k - 6], lw=2.5,
                         ls=['-', '--', ':'][b], alpha=0.9)
axes[2].set_title('bus/phase: phases over steps (red = Q1, blue = Q2, thick = head, grey = idle)', fontsize=9)
axes[2].set_xlabel('step')
fig.tight_layout()
fig.savefig(OUT_DIR / 'busnet_summary.png', dpi=130)


In [ ]:
# optional control: one question on the bus -> nothing to separate, phase should not matter
# N_Q = 1  # then re-run the data + arms cells for 'bus/open msg1' and 'bus/phase msg1'


**How to read it.** The result is positive if `bus/phase msg1` sits above
`bus/open msg1` by a margin that shrinks as `msg_dim` grows, with
`same_q_align` and `h_own` near +1, `cross_q_align` and `h_oth` near zero, `n_clusters_eff` near 2,
and `d_zero` (setting every phase to 0 at test time) costing what the open
bus costs. If `bus/phase` ends at `R` ≈ 1 with `same` ≈ `cross` ≈ 1, the
phases have collapsed to global sync again and the bus is behaving as an
open bus; the stimulus arm tests whether letting a module set its own
phase from its state avoids that.
